In [3]:
import os
import cv2
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from glob import glob
from PIL import Image
import numpy as np

# 1. Face Dataset for Deepfake Detection
class CelebDFDataset(Dataset):
    def __init__(self, video_dir, label, transform=None, num_frames=16):
        self.video_paths = glob(os.path.join(video_dir, '*.mp4'))[:10]  # use only first 10 videos
        self.label = label
        self.transform = transform
        self.num_frames = num_frames

    def __len__(self):
        return len(self.video_paths)

    def read_video(self, path):
        cap = cv2.VideoCapture(path)
        frames = []
        while len(frames) < self.num_frames and cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = Image.fromarray(frame)
            if self.transform:
                frame = self.transform(frame)
            frames.append(frame)
        cap.release()
        while len(frames) < self.num_frames:
            frames.append(torch.zeros_like(frames[0]))  # pad with zeros
        return torch.stack(frames)

    def __getitem__(self, idx):
        path = self.video_paths[idx]
        frames = self.read_video(path)
        label = torch.tensor([self.label], dtype=torch.float32)
        return frames, label

# 2. Xception-like Feature Extractor
class XceptionFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        model = models.mobilenet_v2(pretrained=True)  # lightweight alternative
        self.feature_extractor = nn.Sequential(*list(model.features.children()))

    def forward(self, x):
        B, T, C, H, W = x.size()
        x = x.view(B * T, C, H, W)
        features = self.feature_extractor(x)
        features = torch.nn.functional.adaptive_avg_pool2d(features, (1, 1))
        features = features.view(B, T, -1)
        return features

# 3. Temporal Classifier
class DeepfakeDetector(nn.Module):
    def __init__(self, input_size=1280, hidden_size=256):
        super().__init__()
        self.cnn = XceptionFeatureExtractor()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        features = self.cnn(x)
        _, (h_n, _) = self.lstm(features)
        out = self.classifier(h_n[-1])
        return out

# 4. Transformations
transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 5. Dataset and DataLoader
real_dataset = CelebDFDataset('Celeb-real', label=0, transform=transform)
fake_dataset = CelebDFDataset('Celeb-synthesis', label=1, transform=transform)

full_dataset = real_dataset + fake_dataset
loader = DataLoader(full_dataset, batch_size=2, shuffle=True, num_workers=0)

# 6. Training Loop
model = DeepfakeDetector().cuda()
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(10):
    model.train()
    total_loss = 0
    for i, (frames, labels) in enumerate(loader):
        frames = frames.cuda()
        labels = labels.cuda()
        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss / len(loader):.4f}")


Epoch 1: Loss = 0.7078
Epoch 2: Loss = 0.7338
Epoch 3: Loss = 0.7128
Epoch 4: Loss = 0.7036
Epoch 5: Loss = 0.7196
Epoch 6: Loss = 0.6941
Epoch 7: Loss = 0.6901
Epoch 8: Loss = 0.6830
Epoch 9: Loss = 0.6856
Epoch 10: Loss = 0.7046


In [1]:
import os
import cv2
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from glob import glob
from PIL import Image
import numpy as np
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt

# 1. Face Dataset for Deepfake Detection
class CelebDFDataset(Dataset):
    def __init__(self, video_dir, label, transform=None, num_frames=30):
        self.video_paths = glob(os.path.join(video_dir, '*.mp4'))[:500] 
        self.label = label
        self.transform = transform
        self.num_frames = num_frames

    def __len__(self):
        return len(self.video_paths)

    def read_video(self, path):
        cap = cv2.VideoCapture(path)
        frames = []
        while len(frames) < self.num_frames and cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = Image.fromarray(frame)
            if self.transform:
                frame = self.transform(frame)
            frames.append(frame)
        cap.release()
        while len(frames) < self.num_frames:
            frames.append(torch.zeros_like(frames[0]))  # pad with zeros
        return torch.stack(frames)

    def __getitem__(self, idx):
        path = self.video_paths[idx]
        frames = self.read_video(path)
        label = torch.tensor([self.label], dtype=torch.float32)
        return frames, label

# 2. Xception-like Feature Extractor
class XceptionFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        model = models.mobilenet_v2(pretrained=True)  # lightweight alternative
        self.feature_extractor = nn.Sequential(*list(model.features.children()))

    def forward(self, x):
        B, T, C, H, W = x.size()
        x = x.view(B * T, C, H, W)
        features = self.feature_extractor(x)
        features = torch.nn.functional.adaptive_avg_pool2d(features, (1, 1))
        features = features.view(B, T, -1)
        return features

# 3. Temporal Classifier
class DeepfakeDetector(nn.Module):
    def __init__(self, input_size=1280, hidden_size=256):
        super().__init__()
        self.cnn = XceptionFeatureExtractor()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        features = self.cnn(x)
        _, (h_n, _) = self.lstm(features)
        out = self.classifier(h_n[-1])
        return out

# 4. Transformations
transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ...existing code...

# 5. Dataset and DataLoader
real_dataset = CelebDFDataset('Celeb-real', label=0, transform=transform)
fake_dataset = CelebDFDataset('Celeb-synthesis', label=1, transform=transform)

full_dataset = real_dataset + fake_dataset

from torch.utils.data import random_split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=0)

# 6. Training Loop with Training and Validation Metrics
model = DeepfakeDetector().cuda()
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(10):
    # Training phase
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for frames, labels in train_loader:
        frames = frames.cuda()
        labels = labels.cuda()
        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = (outputs > 0.5).float()
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)
    
    avg_train_loss = train_loss / len(train_loader)
    train_acc = train_correct / train_total
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for frames, labels in val_loader:
            frames = frames.cuda()
            labels = labels.cuda()
            outputs = model(frames)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = (outputs > 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    avg_val_loss = val_loss / len(val_loader)
    val_acc = val_correct / val_total
    
    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Train Acc = {train_acc*100:.2f}% || Val Loss = {avg_val_loss:.4f}, Val Acc = {val_acc*100:.2f}%")

# Save after training
torch.save(model.state_dict(), "deepfake_modell.pth")

# Load before inference
model = DeepfakeDetector().cuda()
model.load_state_dict(torch.load("deepfake_modell.pth"))

y_true = []
y_pred = []
model.eval()
with torch.no_grad():
    for frames, labels in val_loader:
        frames = frames.cuda()
        labels = labels.cuda()
        outputs = model(frames)
        preds = (outputs > 0.5).float()
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Real", "Fake"]))
print(f"Accuracy: {accuracy_score(y_true, y_pred) * 100:.2f}%")


C:\Users\araut1\AppData\Roaming\Python\Python313\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\araut1\AppData\Roaming\Python\Python313\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1: Train Loss = 0.6348, Train Acc = 63.38% || Val Loss = 0.5023, Val Acc = 79.50%
Epoch 2: Train Loss = 0.5245, Train Acc = 76.50% || Val Loss = 0.4434, Val Acc = 80.00%
Epoch 3: Train Loss = 0.4337, Train Acc = 82.00% || Val Loss = 0.3528, Val Acc = 87.50%
Epoch 4: Train Loss = 0.3553, Train Acc = 86.00% || Val Loss = 0.4335, Val Acc = 81.50%
Epoch 5: Train Loss = 0.3091, Train Acc = 87.25% || Val Loss = 0.3790, Val Acc = 84.50%
Epoch 6: Train Loss = 0.3125, Train Acc = 87.75% || Val Loss = 0.6894, Val Acc = 68.50%
Epoch 7: Train Loss = 0.2762, Train Acc = 89.75% || Val Loss = 0.3627, Val Acc = 88.50%
Epoch 8: Train Loss = 0.2460, Train Acc = 91.50% || Val Loss = 0.4331, Val Acc = 84.50%
Epoch 9: Train Loss = 0.2406, Train Acc = 91.88% || Val Loss = 0.3441, Val Acc = 89.50%
Epoch 10: Train Loss = 0.2336, Train Acc = 91.88% || Val Loss = 0.3979, Val Acc = 86.00%

Classification Report:
              precision    recall  f1-score   support

        Real       0.86      0.86      0

In [8]:
def preprocess_video(video_path, transform, num_frames=30):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while len(frames) < num_frames and cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = Image.fromarray(frame)
        if transform:
            frame = transform(frame)
        frames.append(frame)
    cap.release()
    
    # Padding if needed
    while len(frames) < num_frames:
        frames.append(torch.zeros_like(frames[0]))
        
    video_tensor = torch.stack(frames).unsqueeze(0).cuda()  # Add batch dim
    return video_tensor


In [9]:
model = DeepfakeDetector().cuda()
model.load_state_dict(torch.load("deepfake_modell.pth"))
model.eval()

video_tensor = preprocess_video("test_video.mp4", transform)
with torch.no_grad():
    output = model(video_tensor)
    prediction = (output > 0.5).float().item()

print(f"Prediction: {'Fake' if prediction == 1.0 else 'Real'} (Confidence: {output.item():.4f})")


Prediction: Fake (Confidence: 0.5827)


In [ ]:

# Initialize model
model = DeepfakeDetector().cuda()
model.load_state_dict(torch.load("deepfake_modell.pth"))
model.eval()

# Path to videos folder
video_folder = "Celeb-real"

# Initialize counters and accumulators
total_videos = 0
total_real = 0
total_fake = 0
real_confidences = []
fake_confidences = []
max_videos_to_process = 500  # Set the limit to 500 videos

# Process each video in the folder
for video_file in os.listdir(video_folder)[:max_videos_to_process]:  # Limit to first 500 files
    if video_file.endswith(('.mp4', '.avi', '.mov', '.mkv')):  # Add other video formats if needed
        video_path = os.path.join(video_folder, video_file)
        
        try:
            # Preprocess and predict
            video_tensor = preprocess_video(video_path, transform)
            with torch.no_grad():
                output = model(video_tensor)
                confidence = output.item()
                prediction = (output > 0.5).float().item()
            
            total_videos += 1
            
            if prediction == 1.0:  # Fake
                total_fake += 1
                fake_confidences.append(confidence)
            else:  # Real
                total_real += 1
                real_confidences.append(1 - confidence)  # Since real is 0, we take 1-confidence
            
            print(f"Video {total_videos}/{max_videos_to_process}: {video_file}")
            print(f"Prediction: {'Fake' if prediction == 1.0 else 'Real'} (Confidence: {confidence:.4f})")
            print("-" * 50)
            
            # Early exit if we've processed enough videos (in case there are non-video files)
            if total_videos >= max_videos_to_process:
                break
                
        except Exception as e:
            print(f"Error processing {video_file}: {str(e)}")
            continue

# Calculate statistics
avg_fake_confidence = sum(fake_confidences)/len(fake_confidences) if fake_confidences else 0
avg_real_confidence = sum(real_confidences)/len(real_confidences) if real_confidences else 0

# Print summary
print("\n======= SUMMARY STATISTICS =======")
print(f"Total videos processed: {total_videos} (out of {max_videos_to_process} attempted)")
print(f"Total Real videos: {total_real} ({total_real/max(1,total_videos)*100:.2f}%)")
print(f"Total Fake videos: {total_fake} ({total_fake/max(1,total_videos)*100:.2f}%)")
print(f"Average confidence for Real videos: {avg_real_confidence:.4f}")
print(f"Average confidence for Fake videos: {avg_fake_confidence:.4f}")
print("=================================")

Video: id0_0001.mp4
Prediction: Fake (Confidence: 0.5324)
--------------------------------------------------
Video: id0_0002.mp4
Prediction: Real (Confidence: 0.0467)
--------------------------------------------------
Video: id0_0003.mp4
Prediction: Real (Confidence: 0.0703)
--------------------------------------------------
Video: id0_0004.mp4
Prediction: Real (Confidence: 0.0327)
--------------------------------------------------
Video: id0_0005.mp4
Prediction: Real (Confidence: 0.0188)
--------------------------------------------------
Video: id0_0006.mp4
Prediction: Real (Confidence: 0.0106)
--------------------------------------------------
Video: id0_0007.mp4
Prediction: Real (Confidence: 0.0126)
--------------------------------------------------
Video: id0_0008.mp4
Prediction: Real (Confidence: 0.0239)
--------------------------------------------------
Video: id0_0009.mp4
Prediction: Real (Confidence: 0.0109)
--------------------------------------------------
Video: id10_0000.mp

In [9]:


# Initialize model
model = DeepfakeDetector().cuda()
model.load_state_dict(torch.load("deepfake_modell.pth"))
model.eval()

# Path to videos folder
video_folder = "Celeb-synthesis"

# Initialize counters and accumulators
total_videos = 0
total_real = 0
total_fake = 0
real_confidences = []
fake_confidences = []
max_videos_to_process = 500  # Set the limit to 500 videos

# Process each video in the folder
for video_file in os.listdir(video_folder)[:max_videos_to_process]:  # Limit to first 500 files
    if video_file.endswith(('.mp4', '.avi', '.mov', '.mkv')):  # Add other video formats if needed
        video_path = os.path.join(video_folder, video_file)
        
        try:
            # Preprocess and predict
            video_tensor = preprocess_video(video_path, transform)
            with torch.no_grad():
                output = model(video_tensor)
                confidence = output.item()
                prediction = (output > 0.5).float().item()
            
            total_videos += 1
            
            if prediction == 1.0:  # Fake
                total_fake += 1
                fake_confidences.append(confidence)
            else:  # Real
                total_real += 1
                real_confidences.append(1 - confidence)  # Since real is 0, we take 1-confidence
            
            #print(f"Video {total_videos}/{max_videos_to_process}: {video_file}")
            #print(f"Prediction: {'Fake' if prediction == 1.0 else 'Real'} (Confidence: {confidence:.4f})")
            #print("-" * 50)
            
            # Early exit if we've processed enough videos (in case there are non-video files)
            if total_videos >= max_videos_to_process:
                break
                
        except Exception as e:
            print(f"Error processing {video_file}: {str(e)}")
            continue

# Calculate statistics
avg_fake_confidence = sum(fake_confidences)/len(fake_confidences) if fake_confidences else 0
avg_real_confidence = sum(real_confidences)/len(real_confidences) if real_confidences else 0

# Print summary
print("\n======= SUMMARY STATISTICS =======")
print(f"Total videos processed: {total_videos} (out of {max_videos_to_process} attempted)")
print(f"Total Real videos: {total_real} ({total_real/max(1,total_videos)*100:.2f}%)")
print(f"Total Fake videos: {total_fake} ({total_fake/max(1,total_videos)*100:.2f}%)")
print(f"Average confidence for Real videos: {avg_real_confidence:.4f}")
print(f"Average confidence for Fake videos: {avg_fake_confidence:.4f}")
print("=================================")


======= SUMMARY STATISTICS =======
Total videos processed: 500 (out of 500 attempted)
Total Real videos: 27 (5.40%)
Total Fake videos: 473 (94.60%)
Average confidence for Real videos: 0.7593
Average confidence for Fake videos: 0.8331
